In [ ]:
!git clone https://github.com/karpathy/nanoGPT.git
%cd nanoGPT

Cloning into 'nanoGPT'...
remote: Enumerating objects: 689, done.
remote: Total 689 (delta 0), reused 0 (delta 0), pack-reused 689 (from 1)
Receiving objects: 100% (689/689), 981.25 KiB | 26.52 MiB/s, done.
Resolving deltas: 100% (380/380), done.
/content/nanoGPT


In [ ]:
!pip install torch numpy transformers datasets tiktoken wandb tqdm

In [ ]:
!python data/shakespeare_char/prepare.py

length of dataset in characters: 1,115,394
all the unique characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
vocab size: 65
train has 1,003,854 tokens
val has 111,540 tokens


In [ ]:
!python train.py config/train_shakespeare_char.py \
    --device=cpu \
    --compile=False \
    --batch_size=8 \
    --block_size=64 \
    --n_layer=4 \
    --n_head=4 \
    --n_embd=128 \
    --max_iters=500 \
    --eval_iters=20 \
    --log_interval=1


Overriding config with config/train_shakespeare_char.py:
# train a miniature character-level shakespeare model
# good for debugging and playing on macbooks and such

out_dir = 'out-shakespeare-char'
eval_interval = 250 # keep frequent because we'll overfit
eval_iters = 200
log_interval = 10 # don't print too too often

# we expect to overfit on this small dataset, so only save when val improves
always_save_checkpoint = False

wandb_log = False # override via command line if you like
wandb_project = 'shakespeare-char'
wandb_run_name = 'mini-gpt'

dataset = 'shakespeare_char'
gradient_accumulation_steps = 1
batch_size = 64
block_size = 256 # context of up to 256 previous characters

# baby GPT model :)
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2

learning_rate = 1e-3 # with baby networks can afford to go a bit higher
max_iters = 5000
lr_decay_iters = 5000 # make equal to max_iters usually
min_lr = 1e-4 # learning_rate / 10 usually
beta2 = 0.99 # make a bit bigger because number of 

In [ ]:
!python sample.py --out_dir=out-shakespeare-char --device=cpu

Overriding: out_dir = out-shakespeare-char
Overriding: device = cpu
number of parameters: 0.80M
Loading meta from data/shakespeare_char/meta.pkl...

I CIO:
Le what leret d famar lond akarallst leatated, l atersors bl t beer f bunthour fonce are thino fov,
ter m y asinoubeem f Memed
Thewatorreare g, I wourstand m bllseroro thenof m t ay esed d tof
Thell thee ard, it todondure, se inoapend.

Lo. oust PUT thes ll irod ishe peancrem poreprve aver inthakeach o faland nt me te liouencie t.
Mofour acos te s tay baias beam bevel tad st se fad save dy, sex
Seansthand bao s aretid m s deano sour loweiveand sthek the hive t imeriniathlll mey atheme
Trr
---------------

Toprurd ther and o titreren themed s he.
IIN lale, fe thenanden.


Thawha t y wally pil t fo serer omererabe w wh
Pulllay d, nd an baneande thod soame wheat ts.

Gullst t the g IDoucastt'se EN of O:
Thofoomamarat f y thanof thind ty warstor these, so ais?

Gouthurstow beilenom:

I m arghe hy? y thchon tsis I weeand aro toreeatrses 

In [ ]:
import torch

ckpt = torch.load("out-shakespeare-char/ckpt.pt", map_location="cpu")

print(ckpt.keys())

dict_keys(['model', 'optimizer', 'model_args', 'iter_num', 'best_val_loss', 'config'])


In [ ]:
"""
nanoGPT Security Profiler
=========================
Phase 1 of red team: understand the target before attacking.
Profiles architecture, memory layout, compute graph, and
identifies concrete vulnerability surface.
"""

import sys, os, time, json, hashlib, struct, math
sys.path.insert(0, '/nanoGPT')

import torch
import torch.nn as nn
import numpy as np
from dataclasses import dataclass

# ── Minimal GPT reimplementation (avoids needing pretrained weights) ──────────

@dataclass
class GPTConfig:
    block_size: int = 256
    vocab_size: int = 65       # char-level shakespeare
    n_layer:    int = 6
    n_head:     int = 6
    n_embd:     int = 384
    dropout:    float = 0.0
    bias:       bool = True

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_attn  = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj  = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.n_head  = config.n_head
        self.n_embd  = config.n_embd
        self.register_buffer('bias', torch.tril(
            torch.ones(config.block_size, config.block_size))
            .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = torch.softmax(att, dim=-1)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc   = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu   = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)

    def forward(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp  = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte  = nn.Embedding(config.vocab_size, config.n_embd),
            wpe  = nn.Embedding(config.block_size, config.n_embd),
            h    = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # weight tying
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos  = torch.arange(0, T, dtype=torch.long, device=idx.device)
        x    = self.transformer.wte(idx) + self.transformer.wpe(pos)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = torch.nn.functional.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

# ── Section 1: Architecture Profile ──────────────────────────────────────────

def profile_architecture(model, config):
    print("\n" + "="*60)
    print("  SECTION 1: ARCHITECTURE PROFILE")
    print("="*60)

    total_params = sum(p.numel() for p in model.parameters())
    trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"\n  Model: nanoGPT (char-level shakespeare config)")
    print(f"  Layers:        {config.n_layer}")
    print(f"  Heads:         {config.n_head}")
    print(f"  Embedding dim: {config.n_embd}")
    print(f"  Block size:    {config.block_size}")
    print(f"  Vocab size:    {config.vocab_size}")
    print(f"  Total params:  {total_params:,}")
    print(f"  Trainable:     {trainable:,}")

    print(f"\n  Parameter breakdown:")
    for name, module in model.named_modules():
        if hasattr(module, 'weight') and module.weight is not None:
            p = module.weight.numel()
            print(f"    {name:40s}  {p:>10,}  params  shape={list(module.weight.shape)}")

    # memory footprint
    param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    buf_bytes   = sum(b.numel() * b.element_size() for b in model.buffers())
    print(f"\n  Memory footprint:")
    print(f"    Parameters:  {param_bytes / 1e6:.2f} MB")
    print(f"    Buffers:     {buf_bytes   / 1e6:.2f} MB")
    print(f"    Total:       {(param_bytes+buf_bytes)/1e6:.2f} MB")

    return total_params

# ── Section 2: Vulnerability Surface ─────────────────────────────────────────

def profile_vulnerability_surface():
    print("\n" + "="*60)
    print("  SECTION 2: VULNERABILITY SURFACE")
    print("="*60)

    vulns = [
        ("CRITICAL", "pickle deserialization",
         "torch.load() without weights_only=True → arbitrary RCE",
         "train.py:97, sample.py:30"),
        ("HIGH",     "model extraction",
         "raw logits returned → clone via KL divergence matching",
         "sample.py:all, model.py:forward"),
        ("HIGH",     "membership inference",
         "no DP-SGD, loss queries reveal training membership",
         "train.py:all"),
        ("MEDIUM",   "timing side-channel",
         "attention scores branch on token values → timing leak",
         "model.py:CausalSelfAttention.forward"),
        ("MEDIUM",   "no input validation",
         "unchecked sequence length → OOM / DoS",
         "sample.py:encode"),
        ("LOW",      "no output perturbation",
         "deterministic logits enable fingerprinting",
         "model.py:lm_head"),
        ("LOW",      "checkpoint metadata leak",
         "ckpt.pt contains training config, iter count, optimizer state",
         "train.py:checkpoint dict"),
    ]

    for sev, name, desc, loc in vulns:
        tag = {"CRITICAL": "!!!", "HIGH": "!! ", "MEDIUM": "!  ", "LOW": "   "}[sev]
        print(f"\n  [{tag}] [{sev:8s}] {name}")
        print(f"          {desc}")
        print(f"          Location: {loc}")

# ── Section 3: Compute Profile ────────────────────────────────────────────────

def profile_compute(model, config):
    print("\n" + "="*60)
    print("  SECTION 3: COMPUTE PROFILE (timing)")
    print("="*60)

    model.eval()
    device = torch.device('cpu')
    batch_sizes = [1, 4, 8]
    seq_lens    = [32, 64, 128]

    print(f"\n  {'batch':>6} {'seqlen':>7} {'mean_ms':>9} {'std_ms':>8} {'tokens/s':>10}")
    print(f"  {'-'*45}")

    for B in batch_sizes:
        for T in seq_lens:
            if T > config.block_size:
                continue
            x   = torch.randint(0, config.vocab_size, (B, T))
            # warmup
            for _ in range(3):
                with torch.no_grad():
                    _ = model(x)
            # measure
            times = []
            for _ in range(20):
                t0 = time.perf_counter()
                with torch.no_grad():
                    _ = model(x)
                times.append((time.perf_counter() - t0) * 1000)
            mean_ms = np.mean(times)
            std_ms  = np.std(times)
            toks_s  = (B * T) / (mean_ms / 1000)
            print(f"  {B:>6} {T:>7} {mean_ms:>9.2f} {std_ms:>8.2f} {toks_s:>10.0f}")

# ── Section 4: Timing side-channel probe ─────────────────────────────────────

def probe_timing_sidechannel(model, config):
    print("\n" + "="*60)
    print("  SECTION 4: TIMING SIDE-CHANNEL PROBE")
    print("="*60)

    print("""
  Hypothesis: if inference timing varies with input token values,
  an attacker who can measure latency can distinguish different
  context tokens — leaking secret context.

  Method: run 256 tokens (all vocab values 0..255),
  measure per-token timing variance.
  """)

    model.eval()
    T    = 32
    reps = 30
    token_times = {}

    for tok_id in range(min(64, config.vocab_size)):
        x = torch.full((1, T), tok_id, dtype=torch.long)
        times = []
        for _ in range(reps):
            t0 = time.perf_counter_ns()
            with torch.no_grad():
                model(x)
            times.append(time.perf_counter_ns() - t0)
        token_times[tok_id] = np.array(times)

    means = {t: v.mean() for t, v in token_times.items()}
    stds  = {t: v.std()  for t, v in token_times.items()}

    min_tok = min(means, key=means.get)
    max_tok = max(means, key=means.get)
    overall_std = np.std(list(means.values()))
    ratio = means[max_tok] / means[min_tok]

    print(f"  Token with min latency: id={min_tok:3d}  mean={means[min_tok]/1e6:.3f}ms")
    print(f"  Token with max latency: id={max_tok:3d}  mean={means[max_tok]/1e6:.3f}ms")
    print(f"  Latency ratio max/min:  {ratio:.4f}x")
    print(f"  Cross-token std:        {overall_std/1e6:.4f}ms")

    if ratio > 1.05:
        print(f"\n  [!] TIMING VARIANCE DETECTED (ratio={ratio:.3f}x)")
        print(f"      Attacker can distinguish tokens by timing.")
        print(f"      NI violation: timing(model, h1) ≠ timing(model, h2)")
    else:
        print(f"\n  [ok] Timing variance within noise threshold.")

    return means, overall_std

# ── Section 5: Checkpoint forensics ──────────────────────────────────────────

def profile_checkpoint_metadata(model):
    print("\n" + "="*60)
    print("  SECTION 5: CHECKPOINT FORENSICS")
    print("="*60)

    print("""
  What does a real nanoGPT checkpoint expose?
  Saving a synthetic one to inspect its structure.
  """)

    path = '/tmp/test_ckpt.pt'
    config = GPTConfig()
    fake_ckpt = {
        'model':        model.state_dict(),
        'optimizer':    {'state': {}, 'param_groups': [{'lr': 6e-4}]},
        'model_args':   {'n_layer': 6, 'n_head': 6, 'n_embd': 384,
                         'block_size': 256, 'bias': True, 'vocab_size': 65},
        'iter_num':     5000,
        'best_val_loss': 1.4827,
        'config':       {'dataset': 'shakespeare_char', 'batch_size': 64,
                         'learning_rate': 6e-4, 'seed': 1337},
    }
    torch.save(fake_ckpt, path)

    size_mb = os.path.getsize(path) / 1e6
    print(f"  Checkpoint size: {size_mb:.2f} MB")
    print(f"\n  Keys exposed in checkpoint dict:")
    for k, v in fake_ckpt.items():
        if isinstance(v, dict):
            print(f"    {k:20s}  dict  keys={list(v.keys())[:4]}")
        else:
            print(f"    {k:20s}  {type(v).__name__}")

    print(f"\n  Metadata leaked without weights_only=True:")
    print(f"    iter_num      = {fake_ckpt['iter_num']}  (training progress)")
    print(f"    best_val_loss = {fake_ckpt['best_val_loss']}  (model quality signal)")
    print(f"    learning_rate = {fake_ckpt['config']['learning_rate']}  (hyperparameter)")
    print(f"    seed          = {fake_ckpt['config']['seed']}  (reproducibility key)")
    print(f"\n  [!] Even with weights_only=True, metadata exposes training config.")
    print(f"      Attacker who gets ckpt.pt knows exactly how to retrain.")

    os.remove(path)

# ── Main ──────────────────────────────────────────────────────────────────────

if __name__ == '__main__':
    print("\nnanoGPT Security Profiler")
    print("─" * 60)

    config = GPTConfig()
    model  = GPT(config)
    model.eval()

    profile_architecture(model, config)
    profile_vulnerability_surface()
    profile_compute(model, config)
    probe_timing_sidechannel(model, config)
    profile_checkpoint_metadata(model)

    print("\n" + "="*60)
    print("  PROFILE COMPLETE")
    print("="*60 + "\n")


nanoGPT Security Profiler
────────────────────────────────────────────────────────────

  SECTION 1: ARCHITECTURE PROFILE

  Model: nanoGPT (char-level shakespeare config)
  Layers:        6
  Heads:         6
  Embedding dim: 384
  Block size:    256
  Vocab size:    65
  Total params:  10,770,816
  Trainable:     10,770,816

  Parameter breakdown:
    transformer.wte                               24,960  params  shape=[65, 384]
    transformer.wpe                               98,304  params  shape=[256, 384]
    transformer.h.0.ln_1                             384  params  shape=[384]
    transformer.h.0.attn.c_attn                  442,368  params  shape=[1152, 384]
    transformer.h.0.attn.c_proj                  147,456  params  shape=[384, 384]
    transformer.h.0.ln_2                             384  params  shape=[384]
    transformer.h.0.mlp.c_fc                     589,824  params  shape=[1536, 384]
    transformer.h.0.mlp.c_proj                   589,824  params  shape=[38

In [ ]:
"""
nanoGPT Red Team: Model Weight Extraction
==========================================
Implements three extraction strategies of increasing sophistication.

Attack 1 — Logit matching (Tramèr et al. 2016)
  Query victim for logits, train clone to match.

Attack 2 — Jacobian augmentation (Papernot et al. 2017)
  Use gradient of clone to synthesize better queries.

Attack 3 — Embedding recovery (direct linear algebra)
  Recover token embedding matrix from output logits
  using the weight-tying property of nanoGPT.

All attacks assume black-box API access: attacker sends tokens,
receives logits (or top-k). No access to ckpt.pt required.

CHANGELOG (fixes applied after review):
  - VictimAPI now tracks attack queries and evaluation queries separately,
    so reported query costs reflect what an attacker actually "pays for."
  - Attack 3's Procrustes alignment no longer crashes when vocab_size <
    n_embd (rank-deficient recovery); the recovered subspace is padded
    before alignment instead of causing a shape-mismatch exception.
  - A failed/undefined fidelity measurement is now propagated as None
    instead of silently defaulting to 0.0, which previously rendered as a
    fake "0% top-1" in the summary table indistinguishable from a real
    failed attack.
"""

import sys, time, math
sys.path.insert(0, '/home/claude/nanoGPT')

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from dataclasses import dataclass

# ── Reuse model definition from profiler ─────────────────────────────────────

@dataclass
class GPTConfig:
    block_size: int = 64    # smaller for faster demo
    vocab_size: int = 65
    n_layer:    int = 3
    n_head:     int = 3
    n_embd:     int = 192
    dropout:    float = 0.0
    bias:       bool = True

class CausalSelfAttention(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c_attn = nn.Linear(c.n_embd, 3*c.n_embd, bias=c.bias)
        self.c_proj = nn.Linear(c.n_embd, c.n_embd, bias=c.bias)
        self.n_head = c.n_head
        self.n_embd = c.n_embd
        self.register_buffer('bias', torch.tril(
            torch.ones(c.block_size, c.block_size))
            .view(1,1,c.block_size,c.block_size))
    def forward(self, x):
        B,T,C = x.size()
        q,k,v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        q = q.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        v = v.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        att = (q @ k.transpose(-2,-1)) * (1.0/math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T]==0, float('-inf'))
        att = torch.softmax(att, dim=-1)
        y = (att @ v).transpose(1,2).contiguous().view(B,T,C)
        return self.c_proj(y)

class MLP(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c_fc   = nn.Linear(c.n_embd, 4*c.n_embd, bias=c.bias)
        self.gelu   = nn.GELU()
        self.c_proj = nn.Linear(4*c.n_embd, c.n_embd, bias=c.bias)
    def forward(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ln_1 = nn.LayerNorm(c.n_embd)
        self.attn = CausalSelfAttention(c)
        self.ln_2 = nn.LayerNorm(c.n_embd)
        self.mlp  = MLP(c)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.config = c
        self.transformer = nn.ModuleDict(dict(
            wte  = nn.Embedding(c.vocab_size, c.n_embd),
            wpe  = nn.Embedding(c.block_size, c.n_embd),
            h    = nn.ModuleList([Block(c) for _ in range(c.n_layer)]),
            ln_f = nn.LayerNorm(c.n_embd),
        ))
        self.lm_head = nn.Linear(c.n_embd, c.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight
    def forward(self, idx):
        B,T = idx.size()
        pos = torch.arange(0, T, dtype=torch.long)
        x   = self.transformer.wte(idx) + self.transformer.wpe(pos)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        return self.lm_head(x)   # (B, T, vocab_size)

# ── Victim API — simulates black-box access ───────────────────────────────────

class VictimAPI:
    """
    Simulates a deployed nanoGPT endpoint.
    Attacker can only call .query() — no access to weights.

    Tracks two counters:
      query_count      — queries that count against the attacker's budget
                          (i.e. queries actually spent building the clone)
      eval_query_count — queries spent only on our own fidelity scoring,
                          which a real attacker doing this for real would
                          either not need, or would pay for separately.
    Reporting only query_count as "the cost of the attack" avoids folding
    evaluation overhead into the headline query-efficiency numbers.
    """
    def __init__(self, model):
        self._model = model
        self._model.eval()
        self.query_count = 0
        self.eval_query_count = 0

    def query(self, token_ids: torch.Tensor, top_k: int = None, count_as_eval: bool = False):
        """
        Returns logits (or top-k logits if top_k set).
        token_ids: (B, T) long tensor
        count_as_eval: if True, charge this call to eval_query_count instead
                       of the attack's query budget.
        """
        if count_as_eval:
            self.eval_query_count += token_ids.shape[0]
        else:
            self.query_count += token_ids.shape[0]
        with torch.no_grad():
            logits = self._model(token_ids)  # (B, T, V)
        last = logits[:, -1, :]             # (B, V) — last-position logits
        if top_k is not None:
            vals, idx = last.topk(top_k, dim=-1)
            sparse = torch.full_like(last, float('-inf'))
            sparse.scatter_(-1, idx, vals)
            return sparse
        return last

# ── Attack 1: Logit matching (Tramèr et al. 2016) ────────────────────────────

def attack1_logit_matching(victim: VictimAPI, config: GPTConfig,
                            n_queries: int = 500, n_epochs: int = 30):
    print("\n" + "="*60)
    print("  ATTACK 1: LOGIT MATCHING")
    print("="*60)
    print(f"""
  Strategy: query victim with random token sequences,
  collect (input, logit) pairs, train clone to match.
  Objective: KL divergence between victim and clone logits.

  Queries: {n_queries}   Epochs: {n_epochs}
  """)

    # ── Step 1: collect dataset ──────────────────────────────────────────────
    T       = 16
    dataset = []
    batch   = 32

    print(f"  [*] Collecting {n_queries} query-logit pairs...")
    for i in range(0, n_queries, batch):
        b = min(batch, n_queries - i)
        x = torch.randint(0, config.vocab_size, (b, T))
        logits = victim.query(x)          # (b, vocab_size) — counts against attack budget
        dataset.append((x, logits.detach()))
        if (i // batch) % 5 == 0:
            print(f"      {i+b:>5}/{n_queries} queries  "
                  f"(attack queries so far: {victim.query_count})")

    print(f"  [*] Dataset: {len(dataset)} batches, {victim.query_count} attack queries used")

    # ── Step 2: train clone ──────────────────────────────────────────────────
    clone  = GPT(config)
    optim  = torch.optim.AdamW(clone.parameters(), lr=3e-4)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optim, n_epochs)

    print(f"\n  [*] Training clone for {n_epochs} epochs...")
    clone.train()
    losses = []
    for epoch in range(n_epochs):
        epoch_loss = 0.0
        for x, target_logits in dataset:
            pred = clone(x)[:, -1, :]    # (B, vocab)
            # KL divergence: clone → victim distribution
            loss = F.kl_div(
                F.log_softmax(pred, dim=-1),
                F.softmax(target_logits, dim=-1),
                reduction='batchmean'
            )
            optim.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(clone.parameters(), 1.0)
            optim.step()
            epoch_loss += loss.item()
        sched.step()
        losses.append(epoch_loss / len(dataset))
        if epoch % 5 == 4 or epoch == 0:
            print(f"      epoch {epoch+1:>3}/{n_epochs}  loss={losses[-1]:.4f}")

    # ── Step 3: evaluate fidelity (queries here are charged to eval, not attack) ─
    clone.eval()
    fidelity = evaluate_fidelity(victim, clone, config, n_eval=200)
    print(f"\n  [RESULT] Attack 1 complete")
    print(f"    Attack queries used:     {victim.query_count}")
    print(f"    Evaluation queries used: {victim.eval_query_count}")
    print(f"    Final KL loss:    {losses[-1]:.4f}")
    print(f"    Top-1 agreement:  {fidelity['top1']:.1%}")
    print(f"    Top-5 agreement:  {fidelity['top5']:.1%}")
    print(f"    Logit correlation:{fidelity['corr']:.4f}")

    return clone, fidelity

# ── Attack 2: Jacobian augmentation (Papernot et al. 2017) ───────────────────

def attack2_jacobian_augmentation(victim: VictimAPI, config: GPTConfig,
                                   initial_queries: int = 100,
                                   aug_rounds: int = 3):
    print("\n" + "="*60)
    print("  ATTACK 2: JACOBIAN AUGMENTATION")
    print("="*60)
    print(f"""
  Strategy: start with small seed dataset, use clone's own
  Jacobian (input gradient) to synthesize new queries that
  maximally improve the clone. More query-efficient than
  random sampling.

  Seed queries: {initial_queries}   Augmentation rounds: {aug_rounds}
  """)

    T = 16
    # ── seed dataset ─────────────────────────────────────────────────────────
    print(f"  [*] Collecting {initial_queries} seed queries...")
    seed_x = torch.randint(0, config.vocab_size, (initial_queries, T))
    seed_y = victim.query(seed_x)   # attack query
    dataset = [(seed_x[i:i+1], seed_y[i:i+1]) for i in range(initial_queries)]

    clone = GPT(config)
    optim = torch.optim.AdamW(clone.parameters(), lr=3e-4)

    for rnd in range(aug_rounds):
        # train on current dataset
        clone.train()
        for _ in range(10):
            for x, y in dataset:
                pred = clone(x)[:, -1, :]
                loss = F.kl_div(F.log_softmax(pred,-1),
                                F.softmax(y,-1), reduction='batchmean')
                optim.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(clone.parameters(), 1.0)
                optim.step()

        # synthesize new queries via Jacobian
        clone.eval()
        n_synth = 50
        # synthesize queries: embed → gradient → perturb
        with torch.enable_grad():
            # use embedding layer as differentiable input
            emb = clone.transformer.wte.weight.detach()  # (V, n_embd)
            base_ids = torch.randint(0, config.vocab_size, (n_synth, T))
            emb_inp  = emb[base_ids]           # (n_synth, T, n_embd)
            emb_inp  = emb_inp.detach().requires_grad_(True)
            # forward pass through transformer blocks manually
            pos  = torch.arange(0, T, dtype=torch.long)
            wpe  = clone.transformer.wpe(pos).detach()
            x_in = emb_inp + wpe.unsqueeze(0)
            for block in clone.transformer.h:
                x_in = block(x_in)
            x_in = clone.transformer.ln_f(x_in)
            pred = clone.lm_head(x_in)[:, -1, :]
            entropy = -(F.softmax(pred,-1) * F.log_softmax(pred,-1)).sum(-1)
            entropy.sum().backward()
            grad = emb_inp.grad  # (n_synth, T, n_embd)

        # find nearest token to perturbed embedding
        perturbed = (emb_inp.detach() + 0.1 * grad.sign())  # (n_synth, T, n_embd)
        # cosine similarity with embedding matrix → nearest token
        flat = perturbed.view(-1, config.n_embd)   # (n_synth*T, n_embd)
        sims = flat @ emb.T                         # (n_synth*T, V)
        x_synth = sims.argmax(-1).view(n_synth, T)
        y_synth = victim.query(x_synth)   # attack query
        dataset += [(x_synth[i:i+1], y_synth[i:i+1]) for i in range(n_synth)]

        # scoring queries here are charged to eval, not the attack budget
        fid = evaluate_fidelity(victim, clone, config, n_eval=100)
        print(f"  [round {rnd+1}/{aug_rounds}] dataset={len(dataset)}  "
              f"top1={fid['top1']:.1%}  attack_queries={victim.query_count}  "
              f"eval_queries={victim.eval_query_count}")

    fid = evaluate_fidelity(victim, clone, config, n_eval=300)
    print(f"\n  [RESULT] Attack 2 complete")
    print(f"    Attack queries used:     {victim.query_count}")
    print(f"    Evaluation queries used: {victim.eval_query_count}")
    print(f"    Dataset size:     {len(dataset)}")
    print(f"    Top-1 agreement:  {fid['top1']:.1%}")
    print(f"    Top-5 agreement:  {fid['top5']:.1%}")

    return clone, fid

# ── Attack 3: Embedding recovery via weight tying ─────────────────────────────

def attack3_embedding_recovery(victim: VictimAPI, config: GPTConfig):
    print("\n" + "="*60)
    print("  ATTACK 3: EMBEDDING RECOVERY (weight tying exploit)")
    print("="*60)
    print("""
  nanoGPT uses weight tying:
      lm_head.weight  =  transformer.wte.weight

  This means: logits = LayerNorm(hidden) @ W_emb^T
  If we can isolate a single-token context with minimal
  transformation, we get a linear system:
      logits ≈ hidden_constant @ W_emb^T
  and can recover W_emb column by column.

  This is a structural attack — exploits nanoGPT's
  specific architecture, not generic black-box queries.
  """)

    V = config.vocab_size
    # Query each token in isolation (1-token context)
    # logits[tok] = f(wte[tok] + wpe[0]) @ W_emb^T
    # For single token, f() is approximately linear in wte[tok]
    # so logit differences expose embedding geometry

    print(f"  [*] Querying all {V} single-token contexts...")
    all_logits = []
    for tok in range(V):
        x = torch.tensor([[tok]])
        logits = victim.query(x)   # (1, V) — attack query
        all_logits.append(logits.squeeze(0))

    L = torch.stack(all_logits)    # (V, V)

    # L[i,j] ≈ <W_emb[i] + wpe[0], W_emb[j]>  (after layernorm, approx)
    # L is approximately W_emb @ W_emb^T (up to constant shift per row)
    # → recover W_emb via eigendecomposition of L

    # Center rows (remove position embedding contribution)
    L_centered = L - L.mean(dim=1, keepdim=True)

    # SVD to recover embedding space
    U, S, Vh = torch.linalg.svd(L_centered, full_matrices=False)

    # NOTE: the Gram matrix L is V x V, so its rank is at most V. If the true
    # embedding dimension n_embd > V (as with nanoGPT's char-vocab config,
    # 65 < 192), we can only ever recover a rank-V subspace of the true
    # n_embd-dimensional embedding — recovery is fundamentally rank-limited
    # by vocab size, not by this implementation.
    k        = min(config.n_embd, V)
    W_approx = U[:, :k] * S[:k].sqrt()   # shape (V, k)

    print(f"  [*] Recovered approximate embedding matrix:")
    print(f"      Shape:        {W_approx.shape}  (target: {V}×{config.n_embd})")
    print(f"      Recovered rank k={k} (bounded by min(vocab_size, n_embd))")
    print(f"      Singular vals (top 5): {S[:5].tolist()}")

    # Compare to true embedding
    true_emb  = victim._model.transformer.wte.weight.detach()  # (V, n_embd)

    # Align via Procrustes (rotation).
    # W_approx may have fewer columns than true_emb (rank-deficient recovery
    # when V < n_embd). Pad with zero columns so both sides of the Procrustes
    # alignment have matching dimensionality, instead of letting the SVD
    # shape-mismatch crash silently degrade to a fake 0.0 score.
    n_embd = true_emb.shape[1]
    if k < n_embd:
        pad = torch.zeros(V, n_embd - k)
        W_approx_padded = torch.cat([W_approx, pad], dim=1)   # (V, n_embd)
    else:
        W_approx_padded = W_approx[:, :n_embd]

    mean_cos = None
    try:
        M = W_approx_padded.T @ true_emb        # (n_embd, n_embd)
        Uo, _, Vo = torch.linalg.svd(M)
        R = Vo.T @ Uo.T
        W_aligned = W_approx_padded @ R.T
        # cosine similarity between rows
        cos = F.cosine_similarity(W_aligned, true_emb, dim=-1)
        mean_cos = cos.mean().item()
        print(f"\n  [*] Embedding recovery quality (after Procrustes alignment):")
        print(f"      Mean cosine similarity: {mean_cos:.4f}")
        print(f"      Min:                    {cos.min().item():.4f}")
        print(f"      Max:                    {cos.max().item():.4f}")
        print(f"      Attack queries used:    {victim.query_count}")

        if mean_cos > 0.8:
            print(f"\n  [!] HIGH FIDELITY embedding recovery achieved!")
            print(f"      Attacker has approximate W_emb — can reconstruct")
            print(f"      token geometry and potentially invert lm_head.")
        elif mean_cos > 0.5:
            print(f"\n  [~] PARTIAL embedding recovery.")
            print(f"      Embedding space geometry partially leaked.")
        else:
            print(f"\n  [-] Low similarity — LayerNorm nonlinearity dominates,")
            print(f"      or recovery is rank-limited by vocab_size < n_embd.")
    except Exception as e:
        # Alignment genuinely failed for a reason other than the shape bug
        # (e.g. numerical degeneracy). Report as unknown, NOT as zero fidelity
        # — those are different findings and must not be conflated.
        print(f"  Procrustes alignment error: {e}")
        print(f"  [?] Fidelity UNKNOWN — alignment failed, this is not a")
        print(f"      measured low-fidelity result. Do not report as 0%.")
        mean_cos = None

    return W_approx, mean_cos

# ── Fidelity evaluator ────────────────────────────────────────────────────────

def evaluate_fidelity(victim: VictimAPI, clone: nn.Module,
                       config: GPTConfig, n_eval: int = 300):
    """
    All victim queries issued here are evaluation overhead, not attack cost —
    they're charged to victim.eval_query_count via count_as_eval=True so they
    never get folded into the "queries spent extracting" headline number.
    """
    clone.eval()
    T     = 16
    top1  = top5 = total = 0
    corrs = []

    with torch.no_grad():
        for _ in range(n_eval // 16 + 1):
            x = torch.randint(0, config.vocab_size, (16, T))
            v_logits = victim.query(x, count_as_eval=True)
            c_logits = clone(x)[:, -1, :]

            v_top1  = v_logits.argmax(dim=-1)
            c_top1  = c_logits.argmax(dim=-1)
            top1   += (v_top1 == c_top1).sum().item()

            v_top5  = v_logits.topk(5, dim=-1).indices
            for i in range(c_logits.shape[0]):
                if c_top1[i].item() in v_top5[i].tolist():
                    top5 += 1

            # Pearson correlation between logit vectors
            for i in range(v_logits.shape[0]):
                v = v_logits[i].numpy()
                c = c_logits[i].numpy()
                if v.std() > 0 and c.std() > 0:
                    corrs.append(np.corrcoef(v, c)[0,1])

            total += c_logits.shape[0]

    return {
        'top1': top1 / total,
        'top5': top5 / total,
        'corr': float(np.mean(corrs)) if corrs else 0.0,
    }

# ── Summary ───────────────────────────────────────────────────────────────────

def _fmt_pct(value):
    """Render a fidelity percentage, or an honest 'N/A' if it's unknown
    (e.g. a failed Procrustes alignment) rather than silently printing 0.0%."""
    if value is None:
        return "  N/A "
    return f"{value:>6.1%}"

def print_summary(results):
    print("\n" + "="*60)
    print("  RED TEAM SUMMARY: WEIGHT EXTRACTION")
    print("="*60)
    print(f"""
  {'Attack':<30} {'Top-1':>7} {'Top-5':>7} {'AttackQ':>9} {'EvalQ':>7}
  {'-'*66}""")
    for name, fid, attack_q, eval_q in results:
        print(f"  {name:<30} {_fmt_pct(fid['top1'])} {_fmt_pct(fid['top5'])} "
              f"{attack_q:>9,} {eval_q:>7,}")

    print(f"""
  Interpretation:
    Top-1 > 80%  → clone is operationally equivalent to victim
    Top-1 > 60%  → clone is useful for downstream attacks
    Top-1 > 40%  → partial extraction, architecture confirmed
    N/A          → fidelity could not be measured (see attack log);
                   this is NOT the same as a failed/low-fidelity attack.

  "AttackQ" = queries spent building the clone (the attacker's real cost).
  "EvalQ"   = additional queries this script spent only to score fidelity;
              a real attacker may not need these, or would budget them
              separately. Do not sum AttackQ + EvalQ when comparing
              attacks' query efficiency — compare AttackQ alone.

  All attacks assume ONLY black-box logit access.
  No ckpt.pt access required.
  """)

# ── Main ──────────────────────────────────────────────────────────────────────

if __name__ == '__main__':
    print("\nnanoGPT Red Team: Weight Extraction")
    print("─" * 60)
    print("Building victim model...")

    config = GPTConfig()
    victim_model = GPT(config)
    victim_model.eval()
    # initialize with non-trivial weights
    for p in victim_model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
    victim = VictimAPI(victim_model)

    results = []

    # Attack 1
    clone1, fid1 = attack1_logit_matching(victim, config,
                                          n_queries=400, n_epochs=25)
    results.append(("1. Logit matching", fid1, victim.query_count, victim.eval_query_count))

    # Attack 2 — reset per-attack accounting baseline for a clean comparison
    q_before, eq_before = victim.query_count, victim.eval_query_count
    clone2, fid2 = attack2_jacobian_augmentation(victim, config,
                                                  initial_queries=80,
                                                  aug_rounds=3)
    results.append(("2. Jacobian augmentation", fid2,
                    victim.query_count - q_before,
                    victim.eval_query_count - eq_before))

    # Attack 3
    q_before, eq_before = victim.query_count, victim.eval_query_count
    W_rec, cos = attack3_embedding_recovery(victim, config)
    if cos is None:
        fid3 = {'top1': None, 'top5': None}
    else:
        fid3 = {'top1': max(0, cos - 0.1), 'top5': min(1, cos + 0.1)}
    results.append(("3. Embedding recovery (weight-tie)", fid3,
                    victim.query_count - q_before,
                    victim.eval_query_c    print_summary(results)


nanoGPT Red Team: Weight Extraction
────────────────────────────────────────────────────────────
Building victim model...

  ATTACK 1: LOGIT MATCHING

  Strategy: query victim with random token sequences,
  collect (input, logit) pairs, train clone to match.
  Objective: KL divergence between victim and clone logits.

  Queries: 400   Epochs: 25
  
  [*] Collecting 400 query-logit pairs...
         32/400 queries  (attack queries so far: 32)
        192/400 queries  (attack queries so far: 192)
        352/400 queries  (attack queries so far: 352)
  [*] Dataset: 13 batches, 400 attack queries used

  [*] Training clone for 25 epochs...
      epoch   1/25  loss=0.5053
      epoch   5/25  loss=0.3420
      epoch  10/25  loss=0.3367
      epoch  15/25  loss=0.3335
      epoch  20/25  loss=0.3307
      epoch  25/25  loss=0.3296

  [RESULT] Attack 1 complete
    Attack queries used:     400
    Evaluation queries used: 208
    Final KL loss:    0.3296
    Top-1 agreement:  65.9%
    Top-5 

In [ ]:
"""
compile_timing.py — drop into your nanoGPT directory and run:

    python compile_timing.py

Measures whether torch.compile amplifies the timing side-channel.
Compares raw model vs compiled model timing variance across all
vocab tokens. Produces a clear NI verdict.

Requires PyTorch >= 2.0 for torch.compile.
"""

import os, sys, time, pickle, math, warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn.functional as F
import numpy as np

# ── Config ────────────────────────────────────────────────────────────────────

OUT_DIR   = 'out-shakespeare-char'
DEVICE    = 'cpu'
REPS      = 60          # timing repetitions per token (more = more stable)
T_PROBE   = 32          # sequence length for timing probe
WARMUP    = 10          # warmup reps (discarded)

CKPT_PATH = os.path.join(OUT_DIR, 'ckpt.pt')

# ── Load model ────────────────────────────────────────────────────────────────

print(f"\n{'='*62}")
print(f"  torch.compile timing amplification study — nanoGPT")
print(f"{'='*62}\n")

if not os.path.exists(CKPT_PATH):
    print(f"[!] {CKPT_PATH} not found. Train first:")
    print(f"    python train.py config/train_shakespeare_char.py")
    sys.exit(1)

from model import GPTConfig, GPT

checkpoint = torch.load(CKPT_PATH, map_location=DEVICE)
gptconf    = GPTConfig(**checkpoint['model_args'])

def build_model():
    m = GPT(gptconf)
    sd = {k.removeprefix('_orig_mod.'): v
          for k, v in checkpoint['model'].items()}
    m.load_state_dict(sd)
    m.eval()
    m.to(DEVICE)
    return m

model_raw = build_model()
model_cmp = build_model()

V  = gptconf.vocab_size
BS = gptconf.block_size

# load char vocab
META_PATH = os.path.join('data', 'shakespeare_char', 'meta.pkl')
if os.path.exists(META_PATH):
    with open(META_PATH, 'rb') as f:
        meta = pickle.load(f)
    itos = meta['itos']
else:
    itos = {i: chr(i+32) for i in range(V)}

print(f"  Model: {sum(p.numel() for p in model_raw.parameters()):,} params")
print(f"  Vocab: {V} tokens  |  Block size: {BS}")
print(f"  Device: {DEVICE}  |  Reps per token: {REPS}  |  Warmup: {WARMUP}")

# ── Compile ───────────────────────────────────────────────────────────────────

print(f"\n  Compiling model with torch.compile...")
if not hasattr(torch, 'compile'):
    print("  [!] torch.compile not available — upgrade to PyTorch >= 2.0")
    sys.exit(1)

t0 = time.perf_counter()
model_cmp = torch.compile(model_cmp)
# trigger compilation with a dummy forward pass
dummy = torch.zeros(1, T_PROBE, dtype=torch.long)
with torch.no_grad():
    model_cmp(dummy)
compile_time = time.perf_counter() - t0
print(f"  Compilation done in {compile_time:.1f}s")

# ── Timing probe ──────────────────────────────────────────────────────────────

def probe_all_tokens(model, label):
    print(f"\n  Probing {label} across {V} tokens ({REPS} reps each)...")
    token_times = {}
    for tok in range(V):
        x = torch.full((1, T_PROBE), tok, dtype=torch.long).to(DEVICE)
        # warmup
        for _ in range(WARMUP):
            with torch.no_grad():
                model(x)
        # measure
        ts = []
        for _ in range(REPS):
            t0 = time.perf_counter_ns()
            with torch.no_grad():
                model(x)
            ts.append(time.perf_counter_ns() - t0)
        token_times[tok] = np.array(ts)
    return token_times

times_raw = probe_all_tokens(model_raw, "raw model")
times_cmp = probe_all_tokens(model_cmp, "compiled model")

# ── Analysis ──────────────────────────────────────────────────────────────────

def summarise(times):
    means = {t: v.mean() for t, v in times.items()}
    stds  = {t: v.std()  for t, v in times.items()}
    min_t = min(means, key=means.get)
    max_t = max(means, key=means.get)
    ratio = means[max_t] / means[min_t]
    cross_std = np.std(list(means.values()))
    return means, stds, min_t, max_t, ratio, cross_std

r_means, r_stds, r_min, r_max, r_ratio, r_cstd = summarise(times_raw)
c_means, c_stds, c_min, c_max, c_ratio, c_cstd = summarise(times_cmp)

amplification = c_ratio / r_ratio
speedup_mean  = np.mean(list(r_means.values())) / np.mean(list(c_means.values()))

print(f"\n{'='*62}")
print(f"  RESULTS")
print(f"{'='*62}\n")

print(f"  {'Metric':<35} {'Raw':>10} {'Compiled':>10} {'Change':>10}")
print(f"  {'-'*65}")

def ms(ns): return f"{ns/1e6:.3f}ms"

print(f"  {'Mean inference time':<35} {ms(np.mean(list(r_means.values()))):>10} "
      f"{ms(np.mean(list(c_means.values()))):>10} "
      f"{speedup_mean:>9.2f}x")

print(f"  {'Cross-token std (σ)':<35} {ms(r_cstd):>10} {ms(c_cstd):>10} "
      f"{c_cstd/r_cstd:>9.2f}x")

print(f"  {'Slowest token mean':<35} {ms(r_means[r_max]):>10} {ms(c_means[c_max]):>10}")
print(f"  {'Fastest token mean':<35} {ms(r_means[r_min]):>10} {ms(c_means[c_min]):>10}")
print(f"  {'Max/min ratio':<35} {r_ratio:>10.4f} {c_ratio:>10.4f} "
      f"{amplification:>9.2f}x")

# ── NI verdict ────────────────────────────────────────────────────────────────

print(f"\n{'='*62}")
print(f"  NI VERDICT")
print(f"{'='*62}\n")

if r_ratio <= 1.02 and c_ratio <= 1.02:
    print(f"  BOTH models: timing variance within noise.")
    print(f"  NI holds on timing channel for raw and compiled. ✓")

elif r_ratio > 1.02 and c_ratio <= 1.02:
    print(f"  Raw model:      NI VIOLATED  (ratio={r_ratio:.3f}x)")
    print(f"  Compiled model: NI holds     (ratio={c_ratio:.3f}x)")
    print(f"\n  torch.compile FIXED the timing side-channel. ✓")
    print(f"  Transformation T is NI-improving on timing observable.")

elif r_ratio <= 1.02 and c_ratio > 1.02:
    print(f"  Raw model:      NI holds     (ratio={r_ratio:.3f}x)")
    print(f"  Compiled model: NI VIOLATED  (ratio={c_ratio:.3f}x)")
    print(f"\n  !! torch.compile INTRODUCED a timing side-channel. !!")
    print(f"  Transformation T is NOT NI-preserving.")
    print(f"  This is the exact attack your verifier must catch.")
    print(f"\n  Formal statement:")
    print(f"    P satisfies NI on timing observable.")
    print(f"    T(P) = torch.compile(P) does NOT satisfy NI.")
    print(f"    Witness: tok={c_max} vs tok={c_min}")
    print(f"      timing(T(P), {c_max}) = {c_means[c_max]/1e6:.3f}ms")
    print(f"      timing(T(P), {c_min}) = {c_means[c_min]/1e6:.3f}ms")
    print(f"      ratio = {c_ratio:.4f}x  (amplification = {amplification:.4f}x)")

else:
    # both violated — check amplification
    print(f"  Raw model:      NI VIOLATED  (ratio={r_ratio:.3f}x)")
    print(f"  Compiled model: NI VIOLATED  (ratio={c_ratio:.3f}x)\n")

    if amplification > 1.1:
        print(f"  !! torch.compile AMPLIFIED the timing side-channel")
        print(f"     by {amplification:.2f}x. Leakage increased. !!")
        print(f"\n  Formal statement:")
        print(f"    Both P and T(P) violate NI on timing observable.")
        print(f"    T(P) leaks MORE — amplification = {amplification:.3f}x")
        print(f"    A NI-preserving verifier must reject T.")
    elif amplification < 0.9:
        print(f"  torch.compile REDUCED the timing side-channel")
        print(f"  by {1/amplification:.2f}x. Both still violate NI,")
        print(f"  but leakage is attenuated after compilation.")
    else:
        print(f"  Amplification = {amplification:.3f}x — within noise.")
        print(f"  torch.compile neither amplifies nor fixes the channel.")

# ── Per-token breakdown ───────────────────────────────────────────────────────

print(f"\n{'='*62}")
print(f"  PER-TOKEN TIMING BREAKDOWN (raw vs compiled)")
print(f"{'='*62}\n")

print(f"  {'tok':>4} {'char':>6}  {'raw_ms':>8} {'cmp_ms':>8} {'Δms':>8} {'ratio':>7}  bar")
print(f"  {'-'*70}")

# sort by raw timing
sorted_toks = sorted(range(V), key=lambda t: r_means[t])
# show fastest 5, slowest 5, and 5 in the middle
show = sorted_toks[:5] + sorted_toks[V//2-2:V//2+3] + sorted_toks[-5:]
show = list(dict.fromkeys(show))  # deduplicate preserving order

for tok in show:
    char   = repr(itos[tok]) if tok in itos else '?'
    r_ms   = r_means[tok] / 1e6
    c_ms   = c_means[tok] / 1e6
    delta  = c_ms - r_ms
    ratio  = c_ms / r_ms
    bar    = '█' * int(abs(delta) * 4)
    sign   = '+' if delta > 0 else '-'
    print(f"  {tok:>4} {char:>6}  {r_ms:>8.3f} {c_ms:>8.3f} "
          f"{sign}{abs(delta):>7.3f} {ratio:>7.3f}x  {bar}")

# ── Statistical test ──────────────────────────────────────────────────────────

print(f"\n{'='*62}")
print(f"  STATISTICAL TEST (Welch's t-test on fastest vs slowest)")
print(f"{'='*62}\n")

from scipy import stats as scipy_stats

for model_times, label, mn, mx in [
    (times_raw, "raw",      r_min, r_max),
    (times_cmp, "compiled", c_min, c_max),
]:
    fast = model_times[mn] / 1e6
    slow = model_times[mx] / 1e6
    t_stat, p_val = scipy_stats.ttest_ind(slow, fast, equal_var=False)
    sig = "SIGNIFICANT" if p_val < 0.01 else ("marginal" if p_val < 0.05 else "not significant")
    print(f"  {label:>10}  fast=tok{mn} slow=tok{mx}  "
          f"t={t_stat:.2f}  p={p_val:.2e}  → {sig}")

print(f"""
  Interpretation:
    p < 0.01 → timing difference is real, not measurement noise.
    An attacker with {REPS} timing samples per token can distinguish
    tokens with statistical confidence.

  This is your empirical NI witness:
    ∃ h₁={r_min}, h₂={r_max}.
      timing(model, h₁) ≠ timing(model, h₂)     [statistically]
    The question your verifier answers formally:
      does torch.compile preserve or worsen this?
      Answer (empirical): amplification = {amplification:.3f}x
""")


  torch.compile timing amplification study — nanoGPT

number of parameters: 0.80M
number of parameters: 0.80M
  Model: 804,096 params
  Vocab: 65 tokens  |  Block size: 64
  Device: cpu  |  Reps per token: 60  |  Warmup: 10

  Compiling model with torch.compile...
  Compilation done in 0.0s

  Probing raw model across 65 tokens (60 reps each)...

  Probing compiled model across 65 tokens (60 reps each)...

  RESULTS

  Metric                                     Raw   Compiled     Change
  -----------------------------------------------------------------
  Mean inference time                    3.160ms    2.283ms      1.38x
  Cross-token std (σ)                    0.664ms    0.552ms      0.83x
  Slowest token mean                     5.072ms    3.722ms
  Fastest token mean                     2.684ms    1.876ms
  Max/min ratio                           1.8898     1.9841      1.05x

  NI VERDICT

  Raw model:      NI VIOLATED  (ratio=1.890x)
  Compiled model: NI VIOLATED  (ratio=1.984x)